# DeepLense GSoC 2026 — Test I: Multi-Class Classification
**Task:** Classify strong gravitational lensing images into 3 classes:
- `0` — No substructure
- `1` — Subhalo substructure  
- `2` — Vortex substructure

**Strategy:** We train two models and compare:
1. A custom lightweight CNN (designed from scratch for single-channel 150×150 input)
2. EfficientNet-B0 (pretrained, adapted for 1-channel input via transfer learning)

**Evaluation:** ROC curve + AUC score (one-vs-rest for each class)

## 1. Imports & Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Dataset Loading
Expected folder structure after unzipping:
```
dataset/
  no_sub/      → class 0
  subhalo/     → class 1  
  vortex/      → class 2
```
Each sample is a `.npy` file with shape `(1, 150, 150)`, already min-max normalized to [0, 1].

In [ ]:
# ── Update this path to where you unzipped the dataset ──
DATASET_ROOT = './dataset'

CLASS_MAP = {'no_sub': 0, 'subhalo': 1, 'vortex': 2}

def load_dataset(root):
    """Load all .npy files and assign integer labels."""
    paths, labels = [], []
    for class_name, label in CLASS_MAP.items():
        folder = os.path.join(root, class_name)
        files = sorted(glob(os.path.join(folder, '*.npy')))
        paths.extend(files)
        labels.extend([label] * len(files))
        print(f'  {class_name}: {len(files)} samples')
    return paths, labels

print('Loading dataset...')
all_paths, all_labels = load_dataset(DATASET_ROOT)
print(f'Total: {len(all_paths)} samples')

# 90/10 train-test split (stratified)
train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_paths, all_labels, test_size=0.1, stratify=all_labels, random_state=SEED
)
print(f'Train: {len(train_paths)} | Test: {len(test_paths)}')

In [ ]:
class LensingDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths = paths
        self.labels = labels
        self.augment = augment
        self.aug_transforms = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.load(self.paths[idx]).astype(np.float32)  # (1, 150, 150)
        img = torch.from_numpy(img)                         # tensor (1, 150, 150)
        if self.augment:
            img = self.aug_transforms(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

BATCH_SIZE = 64

train_dataset = LensingDataset(train_paths, train_labels, augment=True)
test_dataset  = LensingDataset(test_paths,  test_labels,  augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## 3. Visualize Sample Images

In [ ]:
CLASS_NAMES = ['No Substructure', 'Subhalo', 'Vortex']

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for cls_idx in range(3):
    # pick first sample of each class from train set
    idxs = [i for i, l in enumerate(train_labels) if l == cls_idx]
    for row, sample_idx in enumerate(idxs[:2]):
        img = np.load(train_paths[sample_idx])[0]  # shape (150,150)
        axes[row, cls_idx].imshow(img, cmap='inferno')
        axes[row, cls_idx].set_title(CLASS_NAMES[cls_idx], fontsize=12)
        axes[row, cls_idx].axis('off')

plt.suptitle('Sample Lensing Images per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model A — Custom Lightweight CNN
Designed specifically for single-channel 150×150 scientific images. Uses:
- Progressively deeper conv blocks with BatchNorm + GELU
- Global Average Pooling to reduce parameters
- Dropout for regularization

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=kernel//2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch),
        ) if in_ch != out_ch or stride != 1se nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)


class LensingCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1,  32, stride=2),   # 150 → 75
            ConvBlock(32, 64, stride=2),   # 75  → 38
            ConvBlock(64, 128, stride=2),  # 38  → 19
            ConvBlock(128, 256, stride=2), # 19  → 10
            ConvBlock(256, 512, stride=2), # 10  → 5
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


custom_cnn = LensingCNN(num_classes=3).to(device)
total_params = sum(p.numel() for p in custom_cnn.parameters() if p.requires_grad)
print(f'Custom CNN parameters: {total_params:,}')

## 5. Model B — EfficientNet-B0 (Transfer Learning)
Pretrained on ImageNet, adapted for 1-channel input by replacing the first conv layer.

In [ ]:
def build_efficientnet(num_classes=3):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    # Replace first conv: 3-channel → 1-channel
    old_conv = model.features[0][0]
    model.features[0][0] = nn.Conv2d(
        1, old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )
    # Initialize new layer by averaging pretrained weights across RGB channels
    with torch.no_grad():
        model.features[0][0].weight = nn.Parameter(
            old_conv.weight.mean(dim=1, keepdim=True)
        )

    # Replace classifier head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, num_classes)
    )
    return model


efficientnet = build_efficientnet(num_classes=3).to(device)
total_params = sum(p.numel() for p in efficientnet.parameters() if p.requires_grad)
print(f'EfficientNet-B0 parameters: {total_params:,}')

## 6. Training Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    if scheduler:
        scheduler.step()
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.cpu().numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    return total_loss / total, correct / total, all_probs, all_labels


def train_model(model, model_name, epochs=25, lr=1e-3):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, best_weights = 0.0, None

    print(f'\n{'='*55}')
    print(f'  Training {model_name}')
    print(f'{'='*55}')

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
        va_loss, va_acc, _, _ = evaluate(model, test_loader, criterion)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{epochs} | '
                  f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
                  f'Val Loss: {va_loss:.4f} Acc: {va_acc:.4f}')

    model.load_state_dict(best_weights)
    print(f'\n  ✅ Best Val Accuracy: {best_val_acc:.4f}')
    return history

## 7. Train Both Models

In [ ]:
EPOCHS = 25

history_cnn = train_model(custom_cnn, 'Custom LensingCNN', epochs=EPOCHS, lr=1e-3)
torch.save(custom_cnn.state_dict(), 'custom_cnn_best.pth')

In [ ]:
history_eff = train_model(efficientnet, 'EfficientNet-B0', epochs=EPOCHS, lr=3e-4)
torch.save(efficientnet.state_dict(), 'efficientnet_best.pth')

## 8. Training Curves

In [ ]:
def plot_history(histories, names):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['steelblue', 'darkorange']

    for hist, name, c in zip(histories, names, colors):
        epochs = range(1, len(hist['train_loss']) + 1)
        axes[0].plot(epochs, hist['train_loss'], '--', color=c, alpha=0.6, label=f'{name} train')
        axes[0].plot(epochs, hist['val_loss'],   '-',  color=c,            label=f'{name} val')
        axes[1].plot(epochs, hist['train_acc'],  '--', color=c, alpha=0.6, label=f'{name} train')
        axes[1].plot(epochs, hist['val_acc'],    '-',  color=c,            label=f'{name} val')

    for ax, title, ylabel in zip(axes,
                                  ['Loss', 'Accuracy'],
                                  ['Cross-Entropy Loss', 'Accuracy']):
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.suptitle('Training Curves', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(
    [history_cnn, history_eff],
    ['Custom CNN', 'EfficientNet-B0']
)

## 9. ROC Curves & AUC Scores

In [ ]:
criterion = nn.CrossEntropyLoss()

_, _, probs_cnn, true_labels = evaluate(custom_cnn,  test_loader, criterion)
_, _, probs_eff, _            = evaluate(efficientnet, test_loader, criterion)

# Binarize labels for one-vs-rest ROC
y_bin = label_binarize(true_labels, classes=[0, 1, 2])

def plot_roc(probs, y_bin, model_name, ax, linestyle='-'):
    colors = ['#e74c3c', '#2ecc71', '#3498db']
    macro_auc = []
    for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        macro_auc.append(roc_auc)
        ax.plot(fpr, tpr, color=color, lw=2, linestyle=linestyle,
                label=f'{cls_name} (AUC = {roc_auc:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(f'{model_name}\nMacro AUC = {np.mean(macro_auc):.4f}', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    return np.mean(macro_auc)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

macro_cnn = plot_roc(probs_cnn, y_bin, 'Custom LensingCNN', axes[0])
macro_eff = plot_roc(probs_eff, y_bin, 'EfficientNet-B0',   axes[1])

plt.suptitle('ROC Curves — One-vs-Rest (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Macro-Average AUC')
print(f'   Custom CNN     : {macro_cnn:.4f}')
print(f'   EfficientNet-B0: {macro_eff:.4f}')

## 10. Per-Class AUC Summary Table

In [ ]:
print(f'\n{"Class":<22} {"Custom CNN AUC":>16} {"EfficientNet AUC":>18}')
print('-' * 58)
for i, cls_name in enumerate(CLASS_NAMES):
    fpr_c, tpr_c, _ = roc_curve(y_bin[:, i], probs_cnn[:, i])
    fpr_e, tpr_e, _ = roc_curve(y_bin[:, i], probs_eff[:, i])
    auc_c = auc(fpr_c, tpr_c)
    auc_e = auc(fpr_e, tpr_e)
    print(f'{cls_name:<22} {auc_c:>16.4f} {auc_e:>18.4f}')

print('-' * 58)
print(f'{"Macro Average":<22} {macro_cnn:>16.4f} {macro_eff:>18.4f}')

## 11. Discussion & Strategy

### Data
- Each image is a single-channel (grayscale) 150×150 NumPy array, min-max normalized to [0, 1]
- The three classes differ in subtle substructure patterns — not coarse object-level features
- We applied random flips and rotations as augmentation, which is physically valid since lensing images have no preferred orientation

### Model A — Custom LensingCNN
- Residual ConvBlocks with GELU activations and BatchNorm, designed from scratch for this task
- Progressive downsampling (150 → 5) with Global Average Pooling to avoid overfitting
- Demonstrates understanding of the problem structure without relying on pretrained weights

### Model B — EfficientNet-B0
- Pretrained on ImageNet; the first conv layer was replaced to accept 1-channel input
- Pretrained RGB weights were averaged across channels to initialize the new 1-channel layer — preserving learned low-level feature knowledge
- Lower learning rate (3e-4 vs 1e-3) used because most weights are already pre-trained

### Training Details
- **Loss:** CrossEntropyLoss with label smoothing (0.1) to improve generalization
- **Optimizer:** AdamW with cosine annealing LR schedule
- **Split:** 90% train / 10% test, stratified by class
- **Best weights** saved based on validation accuracy

### Evaluation
- ROC curves computed in a one-vs-rest fashion for each class
- AUC score close to 1.0 indicates strong discriminative ability
- EfficientNet-B0 generally outperforms the custom CNN due to its compound-scaled architecture and pretrained feature representations